#Read Silver


In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = "nyc_taxi_datalake.silver.yellow_taxi"
GOLD_DB      = "nyc_taxi_datalake.gold"

df = spark.table(SILVER_TABLE)

print(f"Silver rows loaded: {df.count():,}")
print(f"Partitions available:")
df.select("_pickup_year", "_pickup_month") \
  .distinct() \
  .orderBy("_pickup_year", "_pickup_month") \
  .show()

In [0]:
def write_gold_table(df, table_name: str):
    full_table_name = f"{GOLD_DB}.{table_name}"

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .saveAsTable(full_table_name)

    print(f"Written: {full_table_name} — {df.count():,} rows")

#Monthly Trip Metrics

In [0]:
monthly = df.groupBy("_pickup_year", "_pickup_month") \
            .agg(
                F.count("*")                             .alias("total_trips"),
                F.round(F.avg("trip_distance"), 2)       .alias("avg_distance_miles"),
                F.round(F.avg("fare_amount"), 2)         .alias("avg_fare_usd"),
                F.round(F.avg("tip_pct"), 2)             .alias("avg_tip_pct"),
                F.round(F.sum("total_amount"), 2)        .alias("total_revenue_usd"),
                F.round(F.avg("trip_duration_min"), 2)   .alias("avg_duration_min"),
                F.round(F.avg("passenger_count"), 2)     .alias("avg_passengers"),
                F.round(F.avg("speed_mph"), 2)           .alias("avg_speed_mph"),
                F.countDistinct("pickup_date")           .alias("active_days")
            ) \
            .orderBy("_pickup_year", "_pickup_month")

write_gold_table(monthly, "monthly_trip_metrics")
monthly.show(truncate=False)

#Hourly Demand Patterns

In [0]:
hourly = df.groupBy("_pickup_year", "pickup_dow", "pickup_hour") \
           .agg(
               F.count("*")                           .alias("total_trips"),
               F.round(F.avg("fare_amount"), 2)       .alias("avg_fare_usd"),
               F.round(F.avg("trip_duration_min"), 2) .alias("avg_duration_min"),
               F.round(F.avg("trip_distance"), 2)     .alias("avg_distance_miles"),
               F.round(F.avg("tip_pct"), 2)           .alias("avg_tip_pct")
           ) \
           .orderBy("_pickup_year", "pickup_dow", "pickup_hour")

write_gold_table(hourly, "hourly_demand_patterns")
hourly.show(truncate=False)

#Zone Performance

In [0]:
zones = df.groupBy(
              "_pickup_year",
              "pu_location_id",
              "pickup_zone",
              "pickup_borough"
          ) \
          .agg(
              F.count("*")                           .alias("total_pickups"),
              F.round(F.sum("total_amount"), 2)      .alias("total_revenue_usd"),
              F.round(F.avg("total_amount"), 2)      .alias("avg_revenue_per_trip"),
              F.round(F.avg("trip_distance"), 2)     .alias("avg_distance_miles"),
              F.round(F.avg("trip_duration_min"), 2) .alias("avg_duration_min"),
              F.round(F.avg("tip_pct"), 2)           .alias("avg_tip_pct"),
              F.round(F.avg("speed_mph"), 2)         .alias("avg_speed_mph")
          ) \
          .orderBy(F.desc("total_revenue_usd"))

write_gold_table(zones, "zone_performance")
zones.show(10, truncate=False)

#Payment Behavior

In [0]:
payment = df.withColumn(
                "payment_method",
                F.when(F.col("payment_type") == 1, "Credit Card")
                 .when(F.col("payment_type") == 2, "Cash")
                 .when(F.col("payment_type") == 3, "No Charge")
                 .when(F.col("payment_type") == 4, "Dispute")
                 .otherwise("Other")
            ) \
            .groupBy("_pickup_year", "_pickup_month", "payment_method") \
            .agg(
                F.count("*")                           .alias("total_trips"),
                F.round(F.sum("total_amount"), 2)      .alias("total_revenue_usd"),
                F.round(F.avg("total_amount"), 2)      .alias("avg_fare_usd"),
                F.round(F.avg("tip_pct"), 2)           .alias("avg_tip_pct"),
                F.round(F.avg("trip_distance"), 2)     .alias("avg_distance_miles")
            ) \
            .orderBy("_pickup_year", "_pickup_month", "payment_method")

write_gold_table(payment, "payment_behavior")
payment.show(truncate=False)

#Verify All Gold Tables

In [0]:
print("=== Monthly Trip Metrics ===")
spark.table(f"{GOLD_DB}.monthly_trip_metrics").show(truncate=False)

print("=== Top 10 Busiest Hours ===")
spark.table(f"{GOLD_DB}.hourly_demand_patterns") \
     .orderBy(F.desc("total_trips")) \
     .show(10, truncate=False)

print("=== Top 10 Zones by Revenue ===")
spark.table(f"{GOLD_DB}.zone_performance") \
     .show(10, truncate=False)

print("=== Payment Methods ===")
spark.table(f"{GOLD_DB}.payment_behavior") \
     .orderBy(F.desc("total_trips")) \
     .show(truncate=False)